**Operation Anomaly Detection — Facebook Prophet**

In [ ]:
from collections import Counter
from numpy._core.defchararray import count
from pandas.core.arrays import period
import pandas as pd
import re
import io
import ast
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import prophet
import warnings
warnings.filterwarnings('ignore')

This project detects hourly traffic anomalies on a specific site using Facebook Prophet, a time series forecasting model that captures seasonal and trend patterns to establish a dynamic expected-behavior baseline.
Detection is driven by the hourly operation count, in addition of the two regressors: number of IPs accessing the operation per hour, and the rate of new IPs — addresses appearing for the first time relative to all historically observed IPs.

In [ ]:
from collections import Counter
from numpy._core.defchararray import count
from pandas.core.arrays import period
import pandas as pd
import re
import io
import ast
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import prophet
import warnings
warnings.filterwarnings('ignore')

with open('/content/demo_data12.csv',encoding='utf-8') as g:
  content =g.read()
  df=pd.read_csv(io.StringIO(content))
  df['TimeGenerated [UTC]']=pd.to_datetime(df['TimeGenerated [UTC]'])
  df['ip_list']=df['Column1'].apply(lambda x: x.split('|') if pd.notna(x) else())
  df['ip_count']=df['ip_list'].apply(len)
  df = df.set_index('TimeGenerated [UTC]')
  df=df.resample('h').sum().fillna(0)
  df = df.drop(columns=['Column1'])
  seen_IPs=set()
  io=set()
  for i in df['ip_list']:
    if not i:
      continue
    io.update(i)
  ioC=Counter(io)
  IP_df=pd.DataFrame(ioC,columns=['i','count'])
  IP_df2=(h for h in IP_df['i'] if IP_df['count']>1)
  seen_IPs.update(IP_df2)
  IP_rate=[]
  for i in df['ip_list']:
    if not i:
      IP_rate.append(0.0)
      continue
    IPf=[IP for IP in i if IP not in seen_IPs]
    IP_rate.append((len(IPf)/len(i)))

  df['NewIPRate']=IP_rate

f=prophet.Prophet(daily_seasonality=True,weekly_seasonality=True,yearly_seasonality=True,seasonality_mode='multiplicative')

dn = pd.DataFrame({
    'ds': df.index,
    'y': df['count_'],
    'IPcount': df['ip_count'],
    'newIPs': df['NewIPRate']
})

f.add_regressor('IPcount')
f.add_regressor('newIPs')

f.fit(dn)

# Predict on the training data itself, as you want to predict the last row of existing data.
forecast=f.predict(dn)
f.plot(forecast)
plt.show()

f.plot_components(forecast)
plt.show()

anomalies=pd.merge(forecast,dn,on='ds',how='left')
anomalies=anomalies[(anomalies['y']>anomalies['yhat_upper'])|(anomalies['y']<anomalies['yhat_lower'])]
print("final anomailes list")
d=anomalies[['ds','y','yhat','yhat_lower','yhat_upper','daily','weekly','yearly','IPcount_y','newIPs_y']]
d.to_csv('final_anomalies.csv')

Prophet is configured to recognize daily, weekly, and yearly seasonality in multiplicative mode, with the IP-based features added as external regressors to enrich the forecast signal beyond raw count trends alone.

In [ ]:
f=prophet.Prophet(daily_seasonality=True,weekly_seasonality=True,yearly_seasonality=True,seasonality_mode='multiplicative')

dn = pd.DataFrame({
    'ds': df.index,
    'y': df['count_'],
    'IPcount': df['ip_count'],
    'newIPs': df['NewIPRate']
})

f.add_regressor('IPcount')
f.add_regressor('newIPs')

f.fit(dn)

Once the model produces a forecasted value and confidence interval for each hour, any observation falling outside those bounds is flagged as an anomaly and exported for further review.

In [ ]:
f.plot_components(forecast)
plt.show()

anomalies=pd.merge(forecast,dn,on='ds',how='left')
anomalies=anomalies[(anomalies['y']>anomalies['yhat_upper'])|(anomalies['y']<anomalies['yhat_lower'])]
print("final anomailes list")
d=anomalies[['ds','y','yhat','yhat_lower','yhat_upper','daily','weekly','yearly','IPcount_y','newIPs_y']]
d.to_csv('final_anomalies.csv')